# Chapter 6: Precognition (Thinking Step by Step)

**Technique:** Chain of Thought (CoT)

**Contents**
* [Lesson: Reasoning Before Answering](#lesson)
* [Exercises](#exercises)
* Example Playground

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Reasoning Before Answering

Complex classification, triage, and decisions with multiple criteria benefit from having Claude reason through the problem *before* committing to a final answer. The technique is simple: ask Claude to write its reasoning inside a `<thinking>` block first, then deliver the answer after.

```js
const PROMPT = `Before answering, reason step by step in <thinking></thinking> tags, then give your final answer.

Classify this log line as DEBUG, INFO, WARN, or ERROR:
[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection`;
```

Why this helps:

* **Accuracy**: the model can't take back tokens once generated. By making it reason first, you let it surface relevant context (category definitions, edge cases) before the decision locks in.
* **Transparency**: you get an audit trail, so you can inspect the reasoning to see *why* Claude chose a label, not just what it chose.
* **Calibration**: if the reasoning is wrong, you can tighten the instructions. The scratchpad is a debugging surface, not just a response.

### Prompt level vs. native adaptive thinking

Modern Claude models (including `claude-sonnet-4-6`) support a native `thinking` parameter at the API level, enabled with the regular request parameter `thinking: { type: "adaptive" }` (not a beta header), that activates the model's internal chain of thought. That is the **model level** version: the model controls when and how much it reasons, and the thinking tokens are returned as a separate block.

What you are doing here is the **prompt level** version: you tell Claude to write a visible scratchpad inside `<thinking>` tags as plain output text. Both approaches improve accuracy on hard tasks; the prompt level version requires no extra API parameters and works on all models, making it the easiest starting point.

### Example: log line severity classification

```js
const PROMPT = `Classify this log line's severity. Reason step by step inside <thinking></thinking> tags, then state the severity on a new line.

[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection`;
```

The model might produce:

```
<thinking>
The pool is exhausted and the system degraded to single connection mode. No crash, the fallback succeeded, but service is degraded. That is a warning condition, not a hard error.
</thinking>
WARN
```

In [ ]:
const PROMPT = `Classify this log line's severity. Reason step by step inside <thinking></thinking> tags, then state the severity (DEBUG / INFO / WARN / ERROR) on a final line with no other text.

[2024-03-15 14:32:01] Connection pool exhausted after 30s, falling back to single connection`;

console.log(await getCompletion(PROMPT));

## Exercises

### Exercise 6.1: Triage a support ticket with visible reasoning

**Scenario**: your support team receives tickets that fall into four categories:

* **(A) Bug report**: the software does something it shouldn't, or fails to do something it should.
* **(B) Feature request**: the user wants new capability.
* **(C) Billing**: questions or problems related to payments, subscriptions, or invoices.
* **(D) Other**: anything that doesn't fit the above.

**Task**: write a prompt that presents all four categories to Claude and asks it to classify the ticket below, letting Claude reason step by step in a `<thinking>` block before it commits to a label.

**Success criteria**: Claude's response contains the correct label `(A)`.

In [ ]:
import { includesAny, grade } from "../lib/grading.js";

const TICKET = "The app crashes with a 500 every time I upload a PNG over 5MB.";
const PROMPT = "[Replace this text - include the four categories and ask Claude to classify the ticket, thinking step by step first]\n" + TICKET;

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => includesAny(text, ["(A)"]);
grade(response, gradeExercise(response));

In [ ]:
import { exercise_6_1_hint } from "../hints.js";
console.log(exercise_6_1_hint);

### Exercise 6.2: Letter only output for an automated router

**Scenario**: the classifier from 6.1 feeds an automated ticket router. The router reads a single character from stdin (the category letter) and fails if it receives anything else. No preamble, no parentheses, no explanation: just `A`, `B`, `C`, or `D`.

**Task**: modify your prompt so Claude outputs *only* the category letter. You can instruct Claude directly ("respond with only the single letter") or enforce it with a structured output (`output_config.format`). Do **not** use an assistant turn prefill, which returns a 400 on Sonnet 4.6+.

**Success criteria**: `response.trim()` is exactly `"A"`.

In [ ]:
import { equalsExact, grade } from "../lib/grading.js";

const TICKET = "The app crashes with a 500 every time I upload a PNG over 5MB.";
const PROMPT = "[Replace this text - classify into A/B/C/D and respond with ONLY the single letter]\n" + TICKET;

const response = await getCompletion(PROMPT);
const gradeExercise = (text) => equalsExact(text.trim(), "A");
grade(response, gradeExercise(response));

In [ ]:
import { exercise_6_2_hint } from "../hints.js";
console.log(exercise_6_2_hint);

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Trying to prefill the assistant turn**: appending `{ role: "assistant", content: "(" }` to the messages array to force a parenthetical label returns a **400 Bad Request** on Sonnet 4.6+. Use an explicit instruction ("respond with only the label in the format (A)") or `output_config.format` instead.
* **Letting preamble leak into the letter only answer**: if Claude adds "The answer is A" instead of just "A", your downstream parser breaks. Be explicit: "Respond with ONLY the single letter. No punctuation, no explanation."
* **Skipping the `<thinking>` block**: if you ask Claude to classify directly without a reasoning step, it may get edge case tickets wrong. The scratchpad is cheap (a few hundred tokens) and reliably improves accuracy on ambiguous inputs.

**Stretch challenge**

Extend Exercise 6.1 to return structured output: use `output_config.format` with a JSON Schema that has a `category` string field (the label letter) and a `reason` string field (one sentence). Verify that `JSON.parse(response).category === "A"`.

**Reflect**: when does a visible `<thinking>` scratchpad help versus hurt? Consider latency (extra tokens = more time), cost (you pay for thinking tokens too), and transparency (is the reasoning useful to a human reviewer or only to the model?). When would you use native adaptive thinking at the API level instead?

## Example Playground

Use the cells below to experiment freely. Try different ticket types, add more categories, or switch to structured output with a `reason` field alongside the category.

In [ ]:
// Experiment: classify a batch of tickets and display each with its reasoning
const TICKETS = [
  "My invoice shows a charge I didn't authorize.",
  "Would love a dark mode option in the dashboard.",
  "Getting a 403 on every API call since yesterday's deploy.",
];

const SYSTEM = `You are a support ticket classifier. Categories:
(A) Bug report
(B) Feature request
(C) Billing
(D) Other

For each ticket, reason step by step in <thinking></thinking> tags, then state the label on a new line.`;

for (const ticket of TICKETS) {
  console.log("--- Ticket:", ticket);
  const response = await getCompletion(ticket, SYSTEM);
  console.log(response);
  console.log();
}